# Data Loading and library imports

In [ ]:
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path
import seaborn as sns
import cv2
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tensorflow import keras

PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "cracked & Non Cracked").is_dir()
)

train = pd.read_parquet(PROJECT_ROOT / "train.parquet")
val = pd.read_parquet(PROJECT_ROOT / "validation.parquet")
test = pd.read_parquet(PROJECT_ROOT / "test.parquet")

X_train = train["image_path"]
y_train = train["label"]

X_val = val["image_path"]
y_val = val["label"]

X_test = test["image_path"]
y_test = test["label"]

# EDA

class weights must be used to balance the classes in the dataset.

In [ ]:
# classes distribution in training, validation, and test sets in the same plot next to each other
plt.figure(figsize=(10, 6))
sns.countplot(x=y_train, color="blue", alpha=0.5, label="Train")
sns.countplot(x=y_val, color="red", alpha=0.5, label="Validation")
sns.countplot(x=y_test, color="green", alpha=0.5, label="Test")
plt.title("Classes Distribution in Train, Validation, and Test Sets")
plt.xlabel("Class")
plt.ylabel("Count")
plt.legend()
plt.show()

In [ ]:
# image_sample size height and width
image_sample = Image.open(PROJECT_ROOT / X_train.iloc[11])

image_sample.size



# finding all images sizes in train, validation, and test sets

image_sizes = []
for image_path in X_train:
    image = Image.open(PROJECT_ROOT / image_path)
    image_sizes.append(image.size)

for image_path in X_val:
    image = Image.open(PROJECT_ROOT / image_path)
    image_sizes.append(image.size)

for image_path in X_test:
    image = Image.open(PROJECT_ROOT / image_path)
    image_sizes.append(image.size)

must be resized to 224 * 224

In [ ]:
train.head()

In [ ]:
# 5 sample image
fig, axes = plt.subplots(1, 5, figsize=(15, 5))
images_sample = [Image.open(PROJECT_ROOT / X_train.iloc[i]) for i in [10, 11, 12, 13, 14]]
for i in images_sample:
    axes[images_sample.index(i)].imshow(i)
    axes[images_sample.index(i)].axis("off")
    axes[images_sample.index(i)].set_title(f'Label: {y_train.iloc[images_sample.index(i)+10]}')


# Preprocessing

In [ ]:
# images sample denoising

for i in images_sample:
    img = cv2.imread(str(PROJECT_ROOT / X_train.iloc[images_sample.index(i)+10]))
    denoised_img = cv2.fastNlMeansDenoisingColored(img, None, 10, 10, 7, 21)
    axes[images_sample.index(i)].imshow(cv2.cvtColor(denoised_img, cv2.COLOR_BGR2RGB))
    axes[images_sample.index(i)].axis("off")
    axes[images_sample.index(i)].set_title(f'Denoised Label: {y_train.iloc[images_sample.index(i)+10]}')
    

In [ ]:
fig, axes = plt.subplots(5, 3, figsize=(15, 25))
for i in images_sample:
    rgb_image = np.array(i.convert("RGB"))
    gray_image = cv2.cvtColor(rgb_image, cv2.COLOR_RGB2GRAY)
    _, binary_image = cv2.threshold(
        gray_image, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )
    
    axes[images_sample.index(i), 0].imshow(rgb_image)
    axes[images_sample.index(i), 0].axis("off")
    axes[images_sample.index(i), 0].set_title(f'RGB Label: {y_train.iloc[images_sample.index(i)+10]}') 
    
    axes[images_sample.index(i), 1].imshow(gray_image, cmap="gray")
    axes[images_sample.index(i), 1].axis("off")
    axes[images_sample.index(i), 1].set_title(f'Gray Label: {y_train.iloc[images_sample.index(i)+10]}')
    
    axes[images_sample.index(i), 2].imshow(binary_image, cmap="gray")
    axes[images_sample.index(i), 2].axis("off")
    axes[images_sample.index(i), 2].set_title(f'Binary Label: {y_train.iloc[images_sample.index(i)+10]}')

In [ ]:
def encode(df):
    paths = [str(PROJECT_ROOT / p) for p in df["image_path"]]
    labels = (df["label"] == "Cracked").astype("int32").values
    return paths, labels


train_paths, train_labels = encode(train)
val_paths, val_labels = encode(val)
test_paths, test_labels = encode(test)

# CNN

### Variables defining

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, regularizers
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

CLASS_NAMES = ["Non Cracked", "Cracked"]

tf.keras.utils.set_random_seed(SEED)

### Encoding

In [ ]:
def encode(df):
    paths = [str(PROJECT_ROOT / p) for p in df["image_path"]]
    labels = (df["label"] == "Cracked").astype("int32").values
    return paths, labels


train_paths, train_labels = encode(train)
val_paths, val_labels = encode(val)
test_paths, test_labels = encode(test)

### class weights

In [ ]:
weights = compute_class_weight(
    class_weight="balanced", classes=np.array([0, 1]), y=train_labels
)
class_weight = {0: float(weights[0]), 1: float(weights[1])}
print("Class weights:", class_weight)

### Resizing

In [ ]:
def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    return img, label


def make_ds(paths, labels, training=False, batch_size=BATCH_SIZE):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(len(paths), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    return ds.batch(batch_size).prefetch(AUTOTUNE)


train_ds = make_ds(train_paths, train_labels, training=True)
val_ds = make_ds(val_paths, val_labels)
test_ds = make_ds(test_paths, test_labels)
train_eval_ds = make_ds(train_paths, train_labels)

### Model

version: 1.0

In [ ]:
def build_cnn_1(dropout=0.4):
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = layers.Rescaling(1.0 / 255)(inputs)

    x = layers.Conv2D(32, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(128, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)

    x = layers.flatten()(x)   
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)

    return keras.Model(inputs, outputs, name="cnn_cracks")

version: 2.0

In [ ]:
def build_cnn_2(dropout=0.4):
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = layers.Rescaling(1.0 / 255)(inputs)
    x = layers.RandomTranslation(0.1, 0.1)(x)
    x = layers.RandomFlip("horizontal_and_vertical")(x)
    x = layers.RandomRotation(0.1)(x)
    x = layers.RandomZoom(0.1)(x)
    x = layers.RandomContrast(0.1)(x)

    x = layers.Conv2D(32, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(128, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)

    x = layers.GlobalAveragePooling2D()(x)    
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)

    return keras.Model(inputs, outputs, name="cnn_cracks")

version: 3.0 plank

In [ ]:
def build_cnn_3(dropout=0.4):
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = layers.Rescaling(1.0 / 255)(inputs)
    x = layers.RandomTranslation(0.1, 0.1)(x)
    x = layers.RandomFlip("horizontal_and_vertical")(x)
    x = layers.RandomRotation(0.1)(x)
    x = layers.RandomZoom(0.1)(x)
    x = layers.RandomContrast(0.1)(x)

    x = layers.Conv2D(32, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(128, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)
    
    x = layers.Conv2D(256, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)

    x = layers.GlobalAveragePooling2D()(x)    
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)

    return keras.Model(inputs, outputs, name="cnn_cracks")

version: 4.0

In [ ]:
def build_cnn_4(n_blocks=4, base_filters=32, dense_units=256, dropout=0.4, max_filters=256,
                aug=0.1, convs_per_block=1, batch_norm=False):
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = layers.Rescaling(1.0 / 255)(inputs)

    if aug > 0:
        x = layers.RandomTranslation(aug, aug)(x)
        x = layers.RandomFlip("horizontal_and_vertical")(x)
        x = layers.RandomRotation(aug)(x)
        x = layers.RandomZoom(aug)(x)
        x = layers.RandomContrast(aug)(x)

    for i in range(n_blocks):
        f = min(base_filters * 2 ** i, max_filters)
        for _ in range(convs_per_block):
            x = layers.Conv2D(f, 3, padding="same", use_bias=not batch_norm)(x)
            if batch_norm:
                x = layers.BatchNormalization()(x)
            x = layers.Activation("relu")(x)
        x = layers.MaxPooling2D()(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(dense_units, activation="relu")(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)
    return keras.Model(inputs, outputs, name="cnn_cracks")


def compile_model(model, lr, weight_decay):
    model.compile(
        optimizer=keras.optimizers.AdamW(learning_rate=lr, weight_decay=weight_decay),
        loss="binary_crossentropy",
        metrics=["accuracy", keras.metrics.Precision(name="precision"),
                 keras.metrics.Recall(name="recall"), keras.metrics.AUC(name="auc")],
    )
    return model

# Training

live training monitoring

In [ ]:
from IPython.display import display


class F1Callback(keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        for prefix in ["", "val_"]:
            p, r = logs[f"{prefix}precision"], logs[f"{prefix}recall"]
            logs[f"{prefix}f1"] = 2 * p * r / (p + r + 1e-7)


class LivePlot(keras.callbacks.Callback):

    def on_train_begin(self, logs=None):
        self.hist, self.handle, self.fig = {}, None, None

    def on_epoch_end(self, epoch, logs=None):
        for k, v in logs.items():
            self.hist.setdefault(k, []).append(float(v))
        best = int(np.argmax(self.hist["val_auc"]))

        fig, axes = plt.subplots(1, 4, figsize=(22, 4))
        for ax, (m, title) in zip(axes[:3], [("loss", "Loss"), ("accuracy", "Accuracy"), ("f1", "F1")]):
            ax.plot(self.hist[m], label="train")
            ax.plot(self.hist[f"val_{m}"], label="val")
            ax.set_title(title)
        axes[3].plot(self.hist["val_precision"], label="val precision")
        axes[3].plot(self.hist["val_recall"], label="val recall")
        axes[3].set_title("Val: precision vs recall")

        for ax in axes:
            ax.axvline(best, color="gray", ls="--", alpha=0.5)
            ax.set_xlabel("epoch")
            ax.legend()
        plt.close(fig)

        self.fig = fig
        if self.handle is None:
            self.handle = display(fig, display_id=True)
        else:
            self.handle.update(fig)

    def on_train_end(self, logs=None):
        if self.fig is not None and mlflow.active_run():
            mlflow.log_figure(self.fig, "training_curves.png")

Mlflow

In [ ]:
import mlflow

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("cracked_cnn")


class MLflowLogger(keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        mlflow.log_metrics({k: float(v) for k, v in logs.items()}, step=epoch)

First trial

In [ ]:
def training(model_name="cnn_base", 
             dropout=0.4, 
             weight_decay=1e-4, 
             lr=1e-3,
             batch_size=BATCH_SIZE, 
             epochs=30):
    
    tr_ds = make_ds(train_paths, train_labels, training=True, batch_size=batch_size)
    v_ds = make_ds(val_paths, val_labels, batch_size=batch_size)


    model = build_cnn_1(dropout)
    model.compile(
        optimizer=keras.optimizers.AdamW(learning_rate=lr, weight_decay=weight_decay),
        loss="binary_crossentropy",
        metrics=["accuracy", keras.metrics.Precision(name="precision"),
                 keras.metrics.Recall(name="recall"), keras.metrics.AUC(name="auc")],
    )

    callbacks = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True),
        keras.callbacks.ModelCheckpoint(f"{model_name}.keras", monitor="val_loss", save_best_only=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=3, min_lr=1e-6),
    ]

    history = model.fit(tr_ds, 
                        validation_data=v_ds, 
                        epochs=epochs,
                        class_weight=class_weight, 
                        callbacks=callbacks, 
                        verbose=2)
    return model, history

model, history = training(model_name="cnn_base", epochs=30)

second trial: remove the flatten layer and add GAP layer instead, adding data augmentation layers, and increase the dropout rate to 0.5

In [ ]:
def training(model_name="cnn_2nd_trial", 
             dropout=0.4, 
             weight_decay=1e-4, 
             lr=1e-3,
             batch_size=BATCH_SIZE, 
             epochs=30):

    tr_ds = make_ds(train_paths, train_labels, training=True, batch_size=batch_size)
    v_ds = make_ds(val_paths, val_labels, batch_size=batch_size)

    with mlflow.start_run(run_name=model_name) as run:
        mlflow.log_params(dict(
            dropout=dropout, weight_decay=weight_decay, lr=lr,
            batch_size=batch_size, epochs=epochs, optimizer="AdamW", img_size=IMG_SIZE,
        ))

        model = build_cnn_2(dropout)
        model.compile(
            optimizer=keras.optimizers.AdamW(learning_rate=lr, weight_decay=weight_decay),
            loss="binary_crossentropy",
            metrics=["accuracy", keras.metrics.Precision(name="precision"),
                     keras.metrics.Recall(name="recall"), keras.metrics.AUC(name="auc")],
        )
        
        callbacks = [
            F1Callback(),
            keras.callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True),
            keras.callbacks.ModelCheckpoint(f"{model_name}.keras", monitor="val_loss", save_best_only=True),
            keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=3, min_lr=1e-6),
            MLflowLogger(),
            LivePlot(),
        ]

        history = model.fit(tr_ds, 
                            validation_data=v_ds, 
                            epochs=epochs,
                            class_weight=class_weight, 
                            callbacks=callbacks, 
                            verbose=1)

        h = history.history
        best = int(np.argmin(h["val_loss"]))
        mlflow.log_metrics({
            "best_epoch": best + 1,
            "best_val_loss": h["val_loss"][best],
            "best_val_accuracy": h["val_accuracy"][best],
            "best_val_auc": h["val_auc"][best],
            "best_val_recall": h["val_recall"][best],
            "best_val_precision": h["val_precision"][best],
            "best_val_f1": h["val_f1"][best],
        })
        mlflow.log_artifact(f"{model_name}.keras")

    return model, history

model, history = training(model_name="cnn_2nd_trial", epochs=30, dropout=0.5)

### Tuning

Third trial(tuning): add more convolutional layers, and decrease the dropout rate to 0.4, and monitor auc

In [ ]:
import optuna
from sklearn.model_selection import train_test_split

TUNE_TRAIN_FRAC = 0.25
TUNE_VAL_FRAC = 0.5

tune_train_paths, _, tune_train_labels, _ = train_test_split(
    train_paths, train_labels, train_size=TUNE_TRAIN_FRAC,
    stratify=train_labels, random_state=SEED)
tune_val_paths, _, tune_val_labels, _ = train_test_split(
    val_paths, val_labels, train_size=TUNE_VAL_FRAC,
    stratify=val_labels, random_state=SEED)

print(len(tune_train_paths), len(tune_val_paths))

In [ ]:

class OptunaPruning(keras.callbacks.Callback):
    def __init__(self, trial, monitor="val_auc"):
        super().__init__()
        self.trial, self.monitor = trial, monitor

    def on_epoch_end(self, epoch, logs=None):
        self.trial.report(logs[self.monitor], step=epoch)
        if self.trial.should_prune():
            self.model.stop_training = True


TRIAL_EPOCHS = 8


def objective(trial):
    keras.backend.clear_session()

    arch = dict(
        n_blocks=trial.suggest_int("n_blocks", 3, 5),
        base_filters=trial.suggest_categorical("base_filters", [32, 48, 64]),
        convs_per_block=trial.suggest_categorical("convs_per_block", [1, 2]),
        batch_norm=trial.suggest_categorical("batch_norm", [False, True]),
        dense_units=trial.suggest_categorical("dense_units", [128, 256]),
        dropout=trial.suggest_float("dropout", 0.2, 0.6),
        aug=trial.suggest_float("aug", 0.0, 0.2, step=0.05),
    )
    lr = trial.suggest_float("lr", 1e-4, 5e-3, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [32, 64])

    tr_ds = make_ds(tune_train_paths, tune_train_labels, training=True, batch_size=batch_size)
    v_ds = make_ds(tune_val_paths, tune_val_labels, batch_size=batch_size)

    with mlflow.start_run(run_name=f"optuna_trial_{trial.number}", nested=True):
        mlflow.log_params({**arch, "optimizer": "adamw", "lr": lr,
                           "weight_decay": weight_decay, "batch_size": batch_size})

        model = compile_model(build_cnn_4(**arch), lr, weight_decay)
        history = model.fit(
            tr_ds, validation_data=v_ds, epochs=TRIAL_EPOCHS,
            class_weight=class_weight, verbose=2,
            callbacks=[
                keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=3),
                OptunaPruning(trial),
                MLflowLogger(),
            ],
        )
        aucs = history.history["val_auc"]
        best_auc = float(max(aucs))
        trial.set_user_attr("best_epoch", int(np.argmax(aucs)) + 1)
        mlflow.log_metric("best_val_auc", best_auc)

    if trial.should_prune():
        raise optuna.TrialPruned()
    return best_auc

starting the study 

In [ ]:
study = optuna.create_study(
    study_name="cnn_cracks_v4",
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=4, n_warmup_steps=2),
    storage="sqlite:///optuna.db",
    load_if_exists=True,
)

if len(study.trials) == 0:
    study.enqueue_trial({
        "n_blocks": 4, "base_filters": 32, "convs_per_block": 1, "batch_norm": False,
        "dense_units": 256, "dropout": 0.4, "aug": 0.1,
        "lr": 1e-3, "weight_decay": 1e-4, "batch_size": 32,
    })

with mlflow.start_run(run_name="optuna_study_v4"):
    study.optimize(objective, n_trials=30,
                   catch=(tf.errors.ResourceExhaustedError,))
    mlflow.log_params({f"best_{k}": v for k, v in study.best_params.items()})
    mlflow.log_metric("best_val_auc", study.best_value)

print("Best val_auc:", study.best_value)
print("Best params:", study.best_params)

version 4.0: implementing the tuning parameters from the previous trial

In [ ]:
best = study.best_params
arch_keys = ["n_blocks", "base_filters", "convs_per_block", "batch_norm",
             "dense_units", "dropout", "aug"]
arch = {k: best[k] for k in arch_keys}

model, history = training(
    model_name="cnn_4th_trial",
    arch=arch,
    lr=best["lr"],
    weight_decay=best["weight_decay"],
    batch_size=best["batch_size"],
    epochs=30,
)

plank

In [ ]:
def training(model_name="cnn_3rd_trial", 
             dropout=0.4, 
             weight_decay=1e-4, 
             lr=1e-3,
             batch_size=BATCH_SIZE, 
             epochs=30):

    tr_ds = make_ds(train_paths, train_labels, training=True, batch_size=batch_size)
    v_ds = make_ds(val_paths, val_labels, batch_size=batch_size)

    with mlflow.start_run(run_name=model_name) as run:
        mlflow.log_params(dict(
            dropout=dropout, weight_decay=weight_decay, lr=lr,
            batch_size=batch_size, epochs=epochs, optimizer="AdamW", img_size=IMG_SIZE,
        ))
        mlflow.log_params({"arch": "4 blocks 32-256, dense256", "monitor": "val_auc"})
        model = build_cnn_3(dropout)
        model.compile(
            optimizer=keras.optimizers.AdamW(learning_rate=lr, weight_decay=weight_decay),
            loss="binary_crossentropy",
            metrics=["accuracy", keras.metrics.Precision(name="precision"),
                     keras.metrics.Recall(name="recall"), keras.metrics.AUC(name="auc")],
        )
        
        callbacks = [
            F1Callback(),
            keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=8, restore_best_weights=True),
            keras.callbacks.ModelCheckpoint(f"{model_name}.keras", monitor="val_auc", mode="max", save_best_only=True),
            keras.callbacks.ReduceLROnPlateau(monitor="val_auc", mode="max", factor=0.3, patience=3, min_lr=1e-6),
            MLflowLogger(),
            LivePlot(),
        ]

        history = model.fit(tr_ds, 
                            validation_data=v_ds, 
                            epochs=epochs,
                            class_weight=class_weight, 
                            callbacks=callbacks, 
                            verbose=1)

        h = history.history
        best = int(np.argmax(h["val_auc"]))
        mlflow.log_metrics({
            "best_epoch": best + 1,
            "best_val_loss": h["val_loss"][best],
            "best_val_accuracy": h["val_accuracy"][best],
            "best_val_auc": h["val_auc"][best],
            "best_val_recall": h["val_recall"][best],
            "best_val_precision": h["val_precision"][best],
            "best_val_f1": h["val_f1"][best],
        })
        mlflow.log_artifact(f"{model_name}.keras")

    return model, history

model, history = training(model_name="cnn_3rd_trial", epochs=30, dropout=0.4)

In [ ]:
# Evaluation

from sklearn.metrics import f1_score

evaluation("cnn_4th_trial", run_id=history.run_id)

best_model = keras.models.load_model("cnn_4th_trial.keras")
val_probs = best_model.predict(val_ds, verbose=0).ravel()
ths = np.arange(0.2, 0.81, 0.02)
f1s = [f1_score(val_labels, val_probs >= t) for t in ths]
best_t = float(ths[int(np.argmax(f1s))])
print(f"best threshold = {best_t:.2f} (val F1 = {max(f1s):.4f})")

evaluation("cnn_4th_trial", threshold=best_t)